In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_path = PROJECT_DIR / "data" / "raw" / "ted_talks.csv"

df = pd.read_csv(raw_path)

print("Original dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

Original dataset loaded successfully!
Rows: 4005
Columns: 19


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


In [5]:
# Check missing values and empty text in each column
missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
text_columns = df.select_dtypes(include=["object", "str"]).columns

missing_report["Empty Strings"] = [
    df[col].astype("string").str.strip().eq("").sum()
    if col in text_columns else 0
    for col in df.columns
]

display(missing_report)

,Missing Values,Missing Percentage (%),Empty Strings
talk_id,0,0.00,0
title,0,0.00,0
speaker_1,0,0.00,0
all_speakers,4,0.10,0
occupations,522,13.03,0
about_speakers,503,12.56,0
views,0,0.00,0
recorded_date,1,0.02,0
published_date,0,0.00,0
event,0,0.00,0


In [6]:
# Check missing and empty values in important recommendation fields
important_columns = ["title", "topics", "description", "transcript", "url"]

for col in important_columns:
    empty_count = df[col].astype("string").str.strip().eq("").sum()
    missing_count = df[col].isna().sum()

    print(f"{col}: Missing = {missing_count}, Empty = {empty_count}")

title: Missing = 0, Empty = 0
topics: Missing = 0, Empty = 0
description: Missing = 0, Empty = 0
transcript: Missing = 0, Empty = 0
url: Missing = 0, Empty = 0


In [7]:
# Document how missing values will be handled
missing_decisions = {
    "comments": "Keep missing values; handle only if needed for engagement analysis.",
    "occupations": "Keep records; use an Unknown label only if needed for speaker analysis.",
    "about_speakers": "Keep records; do not remove talks because speaker details are missing.",
    "all_speakers": "Keep records and investigate missing speaker information if needed.",
    "recorded_date": "Keep the talk; handle the missing date only if time-based analysis is needed.",
    "Recommendation fields": "No missing or empty values found in title, topics, description, transcript, and url."
}

for field, decision in missing_decisions.items():
    print(f"{field}: {decision}")

comments: Keep missing values; handle only if needed for engagement analysis.
occupations: Keep records; use an Unknown label only if needed for speaker analysis.
about_speakers: Keep records; do not remove talks because speaker details are missing.
all_speakers: Keep records and investigate missing speaker information if needed.
recorded_date: Keep the talk; handle the missing date only if time-based analysis is needed.
Recommendation fields: No missing or empty values found in title, topics, description, transcript, and url.


In [8]:
# Check exact duplicate rows, talk IDs, and URLs
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate talk IDs:", df["talk_id"].duplicated().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

Exact duplicate rows: 0
Duplicate talk IDs: 0
Duplicate URLs: 0


### Duplicate Record Handling

- Exact duplicate rows found: 0
- Duplicate talk IDs found: 0
- Duplicate URLs found: 0

**Decision:** No duplicate records were removed because all three checks found zero duplicates. The original dataset remains unchanged.

In [9]:
# Select text fields that need cleaning
text_columns = ["title", "topics", "description", "transcript"]

# Create cleaned copies while preserving original columns
for col in text_columns:
    df[f"{col}_clean"] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Text cleaning completed.")
print("Original columns are preserved.")

Text cleaning completed.
Original columns are preserved.


In [10]:
# Compare original text with cleaned text
display(
    df[["title", "title_clean",
        "topics", "topics_clean",
        "description", "description_clean",
        "transcript", "transcript_clean"]].head(3)
)

,title,title_clean,topics,topics_clean,description,description_clean,transcript,transcript_clean
0,Averting the climate crisis,averting the climate crisis,"['alternative energy', 'cars', 'climate change...","['alternative energy', 'cars', 'climate change...",With the same humor and humanity he exuded in ...,with the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...","thank you so much, chris. and it's truly a gre..."
1,The best stats you've ever seen,the best stats you've ever seen,"['Africa', 'Asia', 'Google', 'demo', 'economic...","['africa', 'asia', 'google', 'demo', 'economic...",You've never seen data presented like this. Wi...,you've never seen data presented like this. wi...,"About 10 years ago, I took on the task to teac...","about 10 years ago, i took on the task to teac..."
2,Simplicity sells,simplicity sells,"['computers', 'entertainment', 'interface desi...","['computers', 'entertainment', 'interface desi...",New York Times columnist David Pogue takes aim...,new york times columnist david pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...","(music: ""the sound of silence,"" simon & garfun..."


It displays the first three talks, showing each original text field beside its cleaned version. This lets you verify that lowercase conversion and whitespace cleaning worked while the original data remains unchanged.

In [11]:
# Check empty values after text cleaning
for col in ["title_clean", "topics_clean", "description_clean", "transcript_clean"]:
    empty_count = df[col].eq("").sum()
    print(f"{col}: {empty_count} empty values")

title_clean: 0 empty values
topics_clean: 0 empty values
description_clean: 0 empty values
transcript_clean: 0 empty values


It counts empty strings in each cleaned column. This helps us confirm that cleaning hasn't left any required recommendation text empty.

In [16]:
# Combine cleaned text fields for recommendations
df["combined_text"] = (
    df["title_clean"] + " " +
    df["topics_clean"] + " " +
    df["description_clean"] + " " +
    df["transcript_clean"]
).str.strip()

print("combined_text column created successfully.")
print("Total rows:", len(df))
print("Empty combined texts:", df["combined_text"].eq("").sum())

display(df[["title", "combined_text"]].head(3))
# Display the complete combined text of the first talk

combined_text column created successfully.
Total rows: 4005
Empty combined texts: 0


,title,combined_text
0,Averting the climate crisis,averting the climate crisis ['alternative ener...
1,The best stats you've ever seen,"the best stats you've ever seen ['africa', 'as..."
2,Simplicity sells,"simplicity sells ['computers', 'entertainment'..."


In [17]:
# Save the cleaned dataset without changing the original CSV
processed_dir = PROJECT_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "ted_talks_cleaned.csv"
df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Saved location:", output_path)
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

Cleaned dataset saved successfully!
Saved location: c:\Users\HARIKA M\TED.py\ted-talks-recommender\data\processed\ted_talks_cleaned.csv
Total rows: 4005
Total columns: 24


In [18]:
# Display all column names
print("Total columns:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

Total columns: 24
1. talk_id
2. title
3. speaker_1
4. all_speakers
5. occupations
6. about_speakers
7. views
8. recorded_date
9. published_date
10. event
11. native_lang
12. available_lang
13. comments
14. duration
15. topics
16. related_talks
17. url
18. description
19. transcript
20. title_clean
21. topics_clean
22. description_clean
23. transcript_clean
24. combined_text


In [19]:
# Verify the saved cleaned dataset
cleaned_df = pd.read_csv(output_path)

print("Original row count:", len(pd.read_csv(raw_path)))
print("Cleaned row count:", len(cleaned_df))
print("Original duplicate rows:", pd.read_csv(raw_path).duplicated().sum())
print("Cleaned duplicate rows:", cleaned_df.duplicated().sum())
print("Total columns in cleaned dataset:", len(cleaned_df.columns))
print("Missing combined_text values:", cleaned_df["combined_text"].isna().sum())
print("Empty combined_text values:", cleaned_df["combined_text"].eq("").sum())

Original row count: 4005
Cleaned row count: 4005
Original duplicate rows: 0
Cleaned duplicate rows: 0
Total columns in cleaned dataset: 24
Missing combined_text values: 0
Empty combined_text values: 0


## Task 3: Data Cleaning and Text Preprocessing Report

### Preprocessing steps
1. Loaded the original dataset containing 4,005 rows and 19 columns.
2. Checked missing values and empty strings in the dataset.
3. Documented missing-value handling decisions. Optional missing metadata was retained.
4. Checked duplicate rows, talk IDs, and URLs. No duplicates were found, so no records were removed.
5. Converted the title, topics, description, and transcript to lowercase and cleaned unnecessary whitespace.
6. Preserved the original text columns and created four cleaned text columns.
7. Created `combined_text` by combining the cleaned title, topics, description, and transcript.
8. Saved the processed dataset as `data/processed/ted_talks_cleaned.csv`.

### Before-and-after summary
- Original dataset: 4,005 rows and 19 columns.
- Cleaned dataset: 4,005 rows and 24 columns.
- Duplicate rows removed: 0.
- Empty combined text values: 0.

### Limitations
- Missing optional metadata was retained rather than filled or deleted.
- Punctuation was preserved to avoid damaging meaningful text.
- The topics field may contain list-format characters that could need further processing before text vectorization.
- The combined text may contain repeated words because it includes multiple text fields.

### Conclusion
The dataset was cleaned and saved without modifying the original CSV. The `combined_text` column is prepared for the next stage of the content-based recommendation system.